# Retail Sales Data Processing and Business Insights
## ABC Retail Solutions — End-to-End Data Engineering Pipeline

---

### Project Overview
This notebook implements a complete data engineering pipeline that:
- Ingests retail transaction data from two source systems
- Cleans and transforms the raw data
- Masks PII (Personal Identifiable Information)
- Calculates business KPIs
- Exports cleaned data for Power BI visualization

### Dataset Summary
| Dataset | Records | Description |
|---------|---------|-------------|
| retail_data1 | 4,243 | Source system 1 transactions |
| retail_data2 | 4,251 | Source system 2 transactions |
| product_details | 10 | Reference/dimension table |

### Pipeline Steps
1. Data Ingestion
2. Remove Duplicates
3. Fix Date Formats
4. Fix Missing Prices
5. Standardize Text Columns
6. Remove Invalid Records
7. PII Masking
8. Revenue Calculation
9. Business KPIs
10. Export Cleaned Data

## Step 1: Import Libraries and Setup Logging
Importing all required Python libraries and configuring 
logging to track each pipeline step.

In [2]:
import pandas as pd
import hashlib
import logging
import os

# LOGGING SETUP
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
log = logging.getLogger()

print("Libraries imported successfully!")
print("Logging configured successfully!")

Libraries imported successfully!
Logging configured successfully!


## Step 2: Data Ingestion
Loading retail transaction data from two Excel source files
and the product dimension table.

### Data Sources:
- **retail_data1.xlsx** — 4,243 transaction records from source system 1
- **retail_data2.xlsx** — 4,251 transaction records from source system 2
- **product_details.xlsx** — 10 reference products with standard prices

### Columns in Transaction Data:
| Column | Description |
|--------|-------------|
| transaction_id | Unique transaction identifier |
| customer_id | Customer identifier |
| customer_name | Customer full name |
| product_id | Product identifier |
| price | Product price |
| product_name | Product name |
| category | Product category |
| purchase_location | Online or Offline |
| city | City of purchase |
| transaction_date | Date of transaction |
| quantity | Units purchased |
| payment_method | Payment type used |
| discount | Discount applied |
| email | Customer email (PII) |
| phone | Customer phone (PII) |
| payment_status | Success or Failed |

In [3]:
# LOAD DATA
log.info("Loading data files...")

df1 = pd.read_excel(r"C:\Users\SHASHANK K R\OneDrive\Desktop\RetailProject\Data\retail_data1.xlsx")
df2 = pd.read_excel(r"C:\Users\SHASHANK K R\OneDrive\Desktop\RetailProject\Data\retail_data2.xlsx")
product_dim = pd.read_excel(r"C:\Users\SHASHANK K R\OneDrive\Desktop\RetailProject\Data\product_details.xlsx")

log.info(f"retail_data1: {df1.shape[0]} rows, {df1.shape[1]} columns")
log.info(f"retail_data2: {df2.shape[0]} rows, {df2.shape[1]} columns")
log.info(f"product_details: {product_dim.shape[0]} rows")

print("\nretail_data1 shape:", df1.shape)
print("retail_data2 shape:", df2.shape)
print("product_details shape:", product_dim.shape)

print("\nretail_data1 columns:")
print(df1.columns.tolist())

print("\nFirst 3 rows of retail_data1:")
df1.head(3)

2026-06-02 12:14:34,468 - INFO - Loading data files...
2026-06-02 12:14:35,693 - INFO - retail_data1: 4243 rows, 16 columns
2026-06-02 12:14:35,695 - INFO - retail_data2: 4251 rows, 16 columns
2026-06-02 12:14:35,696 - INFO - product_details: 10 rows



retail_data1 shape: (4243, 16)
retail_data2 shape: (4251, 16)
product_details shape: (10, 4)

retail_data1 columns:
['transaction_id', 'customer_id', 'customer_name', 'product_id', 'price', 'product_name', 'category', 'purchase_location', 'city', 'transaction_date', 'quantity', 'payment_method', 'discount', 'email', 'phone', 'payment_status']

First 3 rows of retail_data1:


,transaction_id,customer_id,customer_name,product_id,price,product_name,category,purchase_location,city,transaction_date,quantity,payment_method,discount,email,phone,payment_status
0,1,642,Troy Mitchell,108,15000.0,Mixer grinder,Home Appliances,offline,Bangalore,2025-12-06 00:00:00,3,UPI,0.10,Troy60@gmail.com,8385276968,successful
1,2,881,Sarah Guerrero,102,70000.0,Phone,ELEC,online,Bangalore,2025-07-21 00:00:00,5,Card,0.35,SarahGuerrero@yahoo.com,7147248911,successful
2,3,505,Samantha Hull,109,65000.0,Refrigerator,Home Appliances,offline,Chennai,2025-07-11 00:00:00,2,UPI,0.05,SamanthaHull@outlook.com,7415321565,successful


## Step 3: Combine Both Datasets
Merging retail_data1 and retail_data2 into a single 
unified dataset for processing.

Both datasets have identical column structures and 
represent transactions from two different source systems.

In [4]:
# COMBINE BOTH DATASETS
log.info("Combining datasets...")

df = pd.concat([df1, df2], ignore_index=True)

log.info(f"Combined total rows: {df.shape[0]}")

print("retail_data1 rows:", df1.shape[0])
print("retail_data2 rows:", df2.shape[0])
print("Combined total rows:", df.shape[0])
print("\nCombined dataset shape:", df.shape)

2026-06-02 12:14:39,992 - INFO - Combining datasets...
2026-06-02 12:14:40,005 - INFO - Combined total rows: 8494


retail_data1 rows: 4243
retail_data2 rows: 4251
Combined total rows: 8494

Combined dataset shape: (8494, 16)


## Step 4: Remove Duplicate Transactions
Identifying and removing duplicate records from the dataset.

### Duplicate Strategy:
- Duplicates identified by: transaction_id, customer_id, 
  product_id, transaction_date
- Where duplicates exist with both successful and failed 
  payment status, only the **successful** record is kept
- Example: Transaction 6 appears twice — keeping successful record

In [5]:
# REMOVE DUPLICATES
log.info("Removing duplicates...")

before = df.shape[0]

# Sort so successful payments come first
df = df.sort_values('payment_status', ascending=True)

# Drop duplicates keeping first (successful) record
df = df.drop_duplicates(
    subset=['transaction_id', 'customer_id', 'product_id', 'transaction_date'],
    keep='first'
)

after = df.shape[0]
removed = before - after

log.info(f"Removed {removed} duplicate rows. Remaining: {after}")

print(f"Rows before deduplication : {before}")
print(f"Rows after deduplication  : {after}")
print(f"Duplicate rows removed    : {removed}")

2026-06-02 12:14:45,303 - INFO - Removing duplicates...
2026-06-02 12:14:45,320 - INFO - Removed 494 duplicate rows. Remaining: 8000


Rows before deduplication : 8494
Rows after deduplication  : 8000
Duplicate rows removed    : 494


## Step 5: Fix Date Formats
Transaction dates come in multiple inconsistent formats:
- Format 1: MM/DD/YYYY (e.g., 7/21/2025)
- Format 2: MM-DD-YYYY (e.g., 02-19-2026)

All dates are standardized to a single datetime format 
using pandas to_datetime with error handling.

In [6]:
# FIX DATE FORMATS
log.info("Fixing date formats...")

df['transaction_date'] = pd.to_datetime(
    df['transaction_date'],
    dayfirst=False,
    errors='coerce'
)

bad_dates = df['transaction_date'].isna().sum()
log.info(f"Rows with invalid dates: {bad_dates}")

# Drop rows where date could not be parsed
df = df.dropna(subset=['transaction_date'])

print(f"Rows with invalid/unparseable dates dropped: {bad_dates}")
print(f"Remaining rows: {df.shape[0]}")
print(f"\nDate range: {df['transaction_date'].min()} to {df['transaction_date'].max()}")

2026-06-02 12:14:51,646 - INFO - Fixing date formats...
2026-06-02 12:14:51,656 - INFO - Rows with invalid dates: 0


Rows with invalid/unparseable dates dropped: 0
Remaining rows: 8000

Date range: 2025-01-05 00:00:00 to 2026-12-04 00:00:00


## Step 6: Fix Missing Prices
Some transaction records have missing price values.

### Fix Strategy:
- Build a price lookup dictionary from product_details table
- Match using product_id to fill missing prices
- Any rows where price still cannot be filled are dropped

This ensures all revenue calculations are accurate.

In [7]:
# FIX MISSING PRICES
log.info("Fixing missing prices...")

# Check missing prices before fix
missing_before = df['price'].isna().sum()
print(f"Missing prices before fix: {missing_before}")

# Build price lookup from product_dim
price_map = product_dim.set_index('product_id')['price'].to_dict()
print(f"\nPrice lookup map: {price_map}")

# Fill missing prices using product_id
df['price'] = df.apply(
    lambda row: price_map.get(row['product_id'], row['price'])
    if pd.isna(row['price']) else row['price'],
    axis=1
)

missing_after = df['price'].isna().sum()
log.info(f"Rows still missing price after fix: {missing_after}")

# Drop rows where price still missing
df = df.dropna(subset=['price'])

print(f"Missing prices after fix : {missing_after}")
print(f"Remaining rows           : {df.shape[0]}")

2026-06-02 12:15:00,039 - INFO - Fixing missing prices...
2026-06-02 12:15:00,107 - INFO - Rows still missing price after fix: 0


Missing prices before fix: 809

Price lookup map: {101: 250000, 102: 70000, 103: 1300, 104: 8000, 105: 45000, 106: 80000, 107: 50000, 108: 15000, 109: 65000, 110: 45000}
Missing prices after fix : 0
Remaining rows           : 8000


## Step 7: Standardize Text Columns
Raw data contains inconsistent text values that need 
to be standardized for accurate grouping and analysis.

### Issues Fixed:
| Column | Issue | Fix |
|--------|-------|-----|
| category | ELEC, electronics, elec | Electronics |
| category | FURN, furniture, furn | Furniture |
| category | CLOTH, clothing, cloth | Clothing |
| category | HOME, home appliances | Home Appliances |
| product_name | LAPTOP, laptop, Laptop | Laptop (Title Case) |
| city | DELHI, delhi | Delhi (Title Case) |
| payment_method | UPI, upi | Upi (Title Case) |
| payment_status | SUCCESSFUL, Successful | successful (lowercase) |

In [8]:
# STANDARDIZE TEXT COLUMNS
log.info("Standardizing text columns...")

# Fix Category names
category_map = {
    'elec'            : 'Electronics',
    'electronics'     : 'Electronics',
    'furn'            : 'Furniture',
    'furniture'       : 'Furniture',
    'cloth'           : 'Clothing',
    'clothing'        : 'Clothing',
    'home appliances' : 'Home Appliances',
    'home'            : 'Home Appliances',
}

df['category'] = (
    df['category']
    .str.lower()
    .str.strip()
    .map(lambda x: category_map.get(x, x))
)

# Fix other text columns
df['product_name']     = df['product_name'].str.strip().str.title()
df['city']             = df['city'].str.strip().str.title()
df['purchase_location']= df['purchase_location'].str.lower().str.strip()
df['payment_method']   = df['payment_method'].str.strip().str.title()
df['payment_status']   = df['payment_status'].str.lower().str.strip()

log.info("Text columns standardized.")

print("Unique Categories after fix:")
print(df['category'].unique())

print("\nUnique Cities:")
print(df['city'].unique())

print("\nUnique Payment Methods:")
print(df['payment_method'].unique())

print("\nUnique Purchase Locations:")
print(df['purchase_location'].unique())

2026-06-02 12:15:05,873 - INFO - Standardizing text columns...
2026-06-02 12:15:05,896 - INFO - Text columns standardized.


Unique Categories after fix:
['Electronics' 'Home Appliances' 'Furniture' 'Clothing']

Unique Cities:
['Bangalore' 'Chennai' 'Mumbai' 'Hyderabad' 'Delhi']

Unique Payment Methods:
['Cash' 'Card' 'Netbanking' 'Upi']

Unique Purchase Locations:
['online' 'offline']


## Step 8: Remove Invalid Records
Removing records that are not suitable for analysis:

1. **Invalid Quantities** — Rows where quantity is 0 or 
   negative are considered data entry errors
2. **Failed Transactions** — Only successful payment 
   transactions are included in revenue analysis

In [9]:
# REMOVE INVALID QUANTITIES
log.info("Removing invalid quantities...")

before = df.shape[0]
df = df[df['quantity'] > 0]
invalid_qty = before - df.shape[0]
log.info(f"Removed {invalid_qty} rows with invalid quantity.")

# KEEP ONLY SUCCESSFUL PAYMENTS
log.info("Filtering only successful transactions...")

before = df.shape[0]
df = df[df['payment_status'] == 'successful']
failed_removed = before - df.shape[0]
log.info(f"Removed {failed_removed} failed/pending transactions.")

print(f"Invalid quantity rows removed : {invalid_qty}")
print(f"Failed transactions removed   : {failed_removed}")
print(f"Remaining rows                : {df.shape[0]}")

2026-06-02 12:15:12,057 - INFO - Removing invalid quantities...
2026-06-02 12:15:12,066 - INFO - Removed 86 rows with invalid quantity.
2026-06-02 12:15:12,067 - INFO - Filtering only successful transactions...
2026-06-02 12:15:12,074 - INFO - Removed 487 failed/pending transactions.


Invalid quantity rows removed : 86
Failed transactions removed   : 487
Remaining rows                : 7427


## Step 9: PII Masking
Protecting customer Personally Identifiable Information (PII)
as per data privacy requirements.

### Masking Strategy:
| Field | Method | Example |
|-------|--------|---------|
| Email | Partial masking — show first 2 chars only | JohnDoe@gmail.com → Jo****@gmail.com |
| Phone | SHA-256 hashing — irreversible encryption | 9876543210 → a3f8c2d1e4 |

This ensures customer privacy while keeping data 
useful for analytics.

In [10]:
# PII MASKING
log.info("Masking PII data...")

def mask_email(email):
    if pd.isna(email):
        return email
    parts = str(email).split('@')
    if len(parts) == 2:
        return parts[0][:2] + '****@' + parts[1]
    return '****'

def hash_phone(phone):
    if pd.isna(phone):
        return phone
    return hashlib.sha256(str(phone).encode()).hexdigest()[:10]

df['email'] = df['email'].apply(mask_email)
df['phone'] = df['phone'].apply(hash_phone)

log.info("PII masking done.")

print("Sample masked emails:")
print(df['email'].head(5).tolist())

print("\nSample hashed phones:")
print(df['phone'].head(5).tolist())

2026-06-02 12:15:17,776 - INFO - Masking PII data...
2026-06-02 12:15:17,799 - INFO - PII masking done.


Sample masked emails:
['Sa****@yahoo.com', 'Jo****@gmail.com', 'Ki****@outlook.com', 'St****@outlook.com', 'Mi****@gmail.com']

Sample hashed phones:
['9c91f9b1b5', '69cd13926e', '733c555c2b', '33bb31ad60', '896355e05d']


## Step 10: Revenue Calculation
Calculating revenue for each transaction.

### Formula:
**Revenue = Price x Quantity x (1 - Discount)**

### Example:
- Price = Rs.70,000
- Quantity = 2
- Discount = 10% (0.10)
- Revenue = 70,000 x 2 x (1 - 0.10) = **Rs.1,26,000**

In [11]:
# CALCULATE REVENUE
log.info("Calculating revenue...")

df['revenue'] = df['price'] * df['quantity'] * (1 - df['discount'])
df['revenue'] = df['revenue'].round(2)

log.info("Revenue column created.")

print("Revenue column created successfully!")
print(f"\nSample revenue values:")
print(df[['product_name','price','quantity','discount','revenue']].head(5))

2026-06-02 12:15:23,399 - INFO - Calculating revenue...
2026-06-02 12:15:23,403 - INFO - Revenue column created.


Revenue column created successfully!

Sample revenue values:
       product_name     price  quantity  discount   revenue
5694         Laptop  250000.0         1      0.35  162500.0
5731  Mixer Grinder   15000.0         5      0.20   60000.0
5763          Shoes    8000.0         5      0.25   30000.0
5748          Shoes    8000.0         5      0.40   24000.0
5717      Microwave   45000.0         3      0.25  101250.0


## Step 11: Add Date Columns
Extracting additional date components from transaction_date
to enable time-based analysis in Power BI.

### New Columns Added:
- **year** — Transaction year
- **month** — Transaction month number
- **month_name** — Transaction month name
- **quarter** — Transaction quarter (Q1/Q2/Q3/Q4)

In [12]:
# ADD DATE COLUMNS
log.info("Adding date columns...")

df['year']       = df['transaction_date'].dt.year
df['month']      = df['transaction_date'].dt.month
df['month_name'] = df['transaction_date'].dt.strftime('%B')
df['quarter']    = df['transaction_date'].dt.quarter

print("Date columns added successfully!")
print(f"\nYears in data  : {sorted(df['year'].unique())}")
print(f"Months in data : {sorted(df['month'].unique())}")
print(f"Quarters       : {sorted(df['quarter'].unique())}")

2026-06-02 12:15:26,712 - INFO - Adding date columns...


Date columns added successfully!

Years in data  : [np.int32(2025), np.int32(2026)]
Months in data : [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
Quarters       : [np.int32(1), np.int32(2), np.int32(3), np.int32(4)]


## Step 12: Business KPIs
Calculating key performance indicators for business reporting.

### KPIs Calculated:
- Total Revenue
- Total Orders
- Average Order Value
- Total Units Sold
- Revenue by Category
- Revenue by City
- Revenue by Channel (Online vs Offline)
- Revenue by Payment Method
- Top Products by Revenue

In [13]:
# CALCULATE KPIs
log.info("Calculating KPIs...")

total_revenue    = df['revenue'].sum()
total_orders     = df['transaction_id'].nunique()
avg_order_value  = round(total_revenue / total_orders, 2)
total_units_sold = df['quantity'].sum()

revenue_by_category = df.groupby('category')['revenue'].sum().reset_index()
revenue_by_city     = df.groupby('city')['revenue'].sum().reset_index()
revenue_by_month    = df.groupby(['year','month','month_name'])['revenue'].sum().reset_index()
revenue_by_channel  = df.groupby('purchase_location')['revenue'].sum().reset_index()
revenue_by_payment  = df.groupby('payment_method')['revenue'].sum().reset_index()
top_products        = df.groupby('product_name')['revenue'].sum().sort_values(ascending=False).reset_index()

print("="*50)
print("       BUSINESS KPI SUMMARY")
print("="*50)
print(f"  Total Revenue      : Rs.{total_revenue:,.2f}")
print(f"  Total Orders       : {total_orders}")
print(f"  Avg Order Value    : Rs.{avg_order_value:,.2f}")
print(f"  Total Units Sold   : {total_units_sold}")
print("="*50)

print("\nRevenue by Category:")
print(revenue_by_category.to_string(index=False))

print("\nRevenue by City:")
print(revenue_by_city.to_string(index=False))

print("\nRevenue by Channel:")
print(revenue_by_channel.to_string(index=False))

print("\nRevenue by Payment Method:")
print(revenue_by_payment.to_string(index=False))

print("\nTop Products by Revenue:")
print(top_products.to_string(index=False))

2026-06-02 12:15:32,497 - INFO - Calculating KPIs...


       BUSINESS KPI SUMMARY
  Total Revenue      : Rs.1,088,854,335.00
  Total Orders       : 7427
  Avg Order Value    : Rs.146,607.56
  Total Units Sold   : 22125

Revenue by Category:
       category     revenue
       Clothing  16127335.0
    Electronics 627242500.0
      Furniture 229762000.0
Home Appliances 215722500.0

Revenue by City:
     city     revenue
Bangalore 206309860.0
  Chennai 232807540.0
    Delhi 227172180.0
Hyderabad 216955220.0
   Mumbai 205609535.0

Revenue by Channel:
purchase_location     revenue
          offline 541327940.0
           online 547526395.0

Revenue by Payment Method:
payment_method     revenue
          Card 279708585.0
          Cash 276594155.0
    Netbanking 256769240.0
           Upi 275782355.0

Top Products by Revenue:
 product_name     revenue
       Laptop 431825000.0
         Sofa 143172000.0
        Phone 120375500.0
 Refrigerator 111559500.0
 Dining Table  86590000.0
    Microwave  78104250.0
           Tv  75042000.0
Mixer Grinder  

## Step 13: Export Cleaned Data
Saving the final cleaned and transformed dataset to Excel
for loading into Power BI dashboard.

### Final Dataset Summary:
- Input Records  : 8,494 (combined from both sources)
- Output Records : 7,427 (after all cleaning steps)
- Total Columns  : 21
- File Format    : Excel (.xlsx)

In [14]:
# EXPORT CLEANED DATA
log.info("Exporting cleaned data...")

output_path = r"C:\Users\SHASHANK K R\OneDrive\Desktop\RetailProject\Data\cleaned_retail_data.xlsx"
df.to_excel(output_path, index=False)

log.info(f"Cleaned data saved to: {output_path}")
log.info(f"Final dataset: {df.shape[0]} rows and {df.shape[1]} columns")

print(f"Cleaned data exported successfully!")
print(f"Location : {output_path}")
print(f"Rows     : {df.shape[0]}")
print(f"Columns  : {df.shape[1]}")
print(f"\nFinal columns:")
print(df.columns.tolist())

2026-06-02 12:15:41,440 - INFO - Exporting cleaned data...
2026-06-02 12:15:43,865 - INFO - Cleaned data saved to: C:\Users\SHASHANK K R\OneDrive\Desktop\RetailProject\Data\cleaned_retail_data.xlsx
2026-06-02 12:15:43,866 - INFO - Final dataset: 7427 rows and 21 columns


Cleaned data exported successfully!
Location : C:\Users\SHASHANK K R\OneDrive\Desktop\RetailProject\Data\cleaned_retail_data.xlsx
Rows     : 7427
Columns  : 21

Final columns:
['transaction_id', 'customer_id', 'customer_name', 'product_id', 'price', 'product_name', 'category', 'purchase_location', 'city', 'transaction_date', 'quantity', 'payment_method', 'discount', 'email', 'phone', 'payment_status', 'revenue', 'year', 'month', 'month_name', 'quarter']


## Pipeline Complete!

### Data Quality Summary:
| Issue Found | Records Fixed/Removed |
|-------------|----------------------|
| Duplicate transactions | 494 removed |
| Invalid quantities | 86 removed |
| Failed transactions | 487 removed |
| Missing prices | Filled from product_details |
| Mixed date formats | All standardized |
| Inconsistent categories | All standardized |
| PII data | Masked and hashed |

### Final Results:
| Metric | Value |
|--------|-------|
| Input Records | 8,494 |
| Output Records | 7,427 |
| Total Revenue | Rs. 1,08,88,54,335 |
| Total Orders | 7,427 |
| Avg Order Value | Rs. 1,46,607 |
| Total Units Sold | 22,125 |
| Top Product | Laptop |
| Top Category | Electronics |
| Top City | Chennai |

### Next Step:
Cleaned data loaded into **Power BI Dashboard** for 
interactive visualization and business reporting.